In [ ]:
import os
from sentence_transformers import SentenceTransformer, CrossEncoder

############
# import sys
# sys.path.append('/content/drive/MyDrive/Nakamo')
# the above 2 lines were used to locate the code using G.Drive
############

# Import our custom modules
from data_processing.parser import parse_pdfs_in_folder
from data_processing.chunker import chunk_data
from vector_store.db_manager import clear_vector_store, create_and_populate_vector_store
from llm.local_llm import LocalLLM
from rag_agents.rag_pipelines import (
    get_rag_response
)


### Configurations

In [ ]:
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "..."
pdf_folder = 'documents'

# --- Retrieval models Configuration ---
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
# --- LLM Configuration ---
LLM_MODEL_ID = "Qwen/Qwen3-0.6B-GGUF"
LLM_MODEL_FILE = "Qwen3-0.6B-Q8_0.gguf"



#### Load the models

In [ ]:

# load the models
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

reranker = CrossEncoder(RERANKER_MODEL)

local_llm = LocalLLM(LLM_MODEL_ID, LLM_MODEL_FILE, verbose=False)


## Document Processing and DB populating

#### Parse the pdf files

In [ ]:
pages_data = parse_pdfs_in_folder(pdf_folder)

#### Chunk the parsed data

In [ ]:
if pages_data:
    chunked_data = chunk_data(pages_data, chunk_size = 1024, chunk_overlap = 256)
    # Print a sample of the output to verify
    if chunked_data:
        print("\n--- Sample of Chunked Data ---")
        sample_chunk = chunked_data[10] # Picking a random chunk
        print(f"Source: {sample_chunk['metadata']['source']}")
        print(f"Page: {sample_chunk['metadata']['page']}")
        print(f"Chunk Text: '{sample_chunk['text']}'")
        print("---------------------------------")


In [ ]:
# clear_vector_store(CHROMA_PATH, COLLECTION_NAME)

#### Populate the DB

In [ ]:
if chunked_data:
    create_and_populate_vector_store(chunked_data, CHROMA_PATH, COLLECTION_NAME, embedding_model)

## Retrieve information based on User Queries.

In [ ]:
user_query = "Hello"

get_rag_response(
    user_query=user_query,
    local_llm=local_llm,
    embedding_model=embedding_model,
    reranker=reranker,
    CHROMA_PATH=CHROMA_PATH,
    COLLECTION_NAME=COLLECTION_NAME,
    final_n_results=5,
    initial_fetch=15,
    max_new_tokens=512,
    rag_temp=0.05,
    direct_temp=0.3
)


In [ ]:
user_query = "What are the design control requirements for verification and validation?"

get_rag_response(
    user_query=user_query,
    local_llm=local_llm,
    embedding_model=embedding_model,
    reranker=reranker,
    CHROMA_PATH=CHROMA_PATH,
    COLLECTION_NAME=COLLECTION_NAME,
    final_n_results=5,
    initial_fetch=15,
    max_new_tokens=512,
    rag_temp=0.05,
    direct_temp=0.3
)
